# ClaimGuard — run everything on Colab

Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.
Each step writes JSON files in `results/`, so if Colab disconnects I only re-run the step that was cut.

There is no school GPU, so everything runs on the free Colab T4. The free quota is limited per day,
so the notebook is cut in **3 sessions** (one per day is fine) and saves everything to Google Drive after each step.
If Colab disconnects, re-run section 0, then go to the step that was cut.

| Session | Sections | Rough time on a T4 |
| --- | --- | --- |
| 1 | 0 → 3 (setup, data, retrieval, baselines) | about 30 min |
| 2 | 4 → 5 (12-run sweep, then 3 seeds) | about 45 min |
| 3 | 6 → 11 (pipeline, ablations, LLM judge, stress tests, LLM-AggreFact) | about 2 h |

Before session 3, do the Hugging Face step of section 9b (account + accept the LLM-AggreFact conditions + token): it takes 5 minutes and the dataset is gated.

## 0. Setup

In [ ]:
# Option A: clone my repo (put the right URL). Option B: upload claimguard.zip in the file panel.
import os
REPO_URL = ""   # ex: https://github.com/<me>/claimguard.git
if REPO_URL:
    !git clone {REPO_URL} claimguard
elif os.path.exists("claimguard.zip"):
    !unzip -q -o claimguard.zip
%cd claimguard
!pip install -q -r requirements-dev.txt && pip install -q -e .

In [ ]:
# keep results/ and models/ on Google Drive so nothing is lost when Colab disconnects
from google.colab import drive
drive.mount("/content/drive")
import shutil
from pathlib import Path
SAVE = Path("/content/drive/MyDrive/claimguard_runs")

def copy_tree(src, dst, overwrite):
    # copy every file of src into dst (keep existing files when overwrite=False)
    for f in Path(src).rglob("*"):
        if f.is_file():
            target = Path(dst) / f.relative_to(src)
            if overwrite or not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)

# restore what earlier sessions saved (does nothing the first time)
for folder in ("results", "models"):
    if (SAVE / folder).exists():
        copy_tree(SAVE / folder, folder, overwrite=False)

def backup():
    for folder in ("results", "models"):
        copy_tree(folder, SAVE / folder, overwrite=True)
    print("saved to Drive")

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU, change the runtime")

In [ ]:
# the tests take about 1 minute and need no download: if they fail, stop here
!make test

## 1. Data

In [ ]:
!make data
!cat data/processed/scifact/stats.json

## 2. Retrieval alone (ablation A1 + reranker) and no-evidence baselines

In [ ]:
!python scripts/evaluate.py retrieval --config configs/default.yaml
!python scripts/evaluate.py baselines --config configs/default.yaml
backup()

## 3. Baselines that use evidence, before any fine-tuning

- rule-based lexical verifier (same retrieval as the main system)
- the minimal system: BM25, no reranker, public NLI checkpoint without fine-tuning
- zero-shot NLI with the full retrieval (this one is also the first point of ablation A5)

In [ ]:
ZS = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
!python scripts/evaluate.py pipeline --config configs/lexical_baseline.yaml --tag lexical
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model {ZS} --retrieval-kind bm25 --rerank-kind none --tag bm25_zeroshot
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model {ZS} --tag zeroshot

## 4. Hyperparameter search (worth 3 points) — 12 short trainings, about 3 min each

In [ ]:
!python scripts/train_verifier.py --config configs/train_verifier.yaml --sweep
!rm -rf models/sweep_tmp   # temporary checkpoint, 700 MB, no need to keep it
backup()
import json; rows = json.load(open("results/sweep.json"))
sorted(rows, key=lambda r: -r["dev_macro_f1"])[:5]

If the best values are different from `configs/train_verifier.yaml` (lr 2e-5, batch_size 16, epochs 5), I change the file **now**:
double-click `claimguard/configs/train_verifier.yaml` in the Files panel, edit the three numbers under `train:`, Ctrl+S. Then run the next cell.

## 5. Train the verifier with 3 seeds

In [ ]:
!python scripts/train_verifier.py --config configs/train_verifier.yaml --all-seeds
backup()

## 6. Calibration, then the main pipeline with all the ablations

In [ ]:
# calibration first: it writes models/calibration.json (temperature + threshold) used by the pipeline
!python scripts/evaluate.py calibration --config configs/default.yaml
!python scripts/evaluate.py pipeline --config configs/default.yaml
# the three seeds, all without temperature scaling so they are comparable -> mean +- std
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model models/verifier --tag seed13
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model models/verifier-seed42 --tag seed42
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model models/verifier-seed2026 --tag seed2026

## 7. Ablation A5: fine-tuning from a plain encoder (no NLI pre-training)

In [ ]:
!python scripts/train_verifier.py --config configs/train_verifier.yaml --base-model microsoft/deberta-v3-base --out-dir models/verifier-plain
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-model models/verifier-plain --tag plain_encoder
backup()

## 8. LLM judge baseline (open-source, zero-shot, quantized)

Qwen2.5-7B in 4-bit takes about 5.5 GB, so it fits on the T4 (16 GB). If it runs out of memory or is too slow,
use `Qwen/Qwen2.5-3B-Instruct` instead (the professor suggested a smaller or quantized model) and write it in the report.

In [ ]:
!pip install -q bitsandbytes accelerate
LLM = "Qwen/Qwen2.5-7B-Instruct"   # fallback: "Qwen/Qwen2.5-3B-Instruct"
!python scripts/evaluate.py pipeline --config configs/default.yaml --verifier-kind llm --verifier-model {LLM} --load-in-4bit --tag llm_judge
backup()

## 9. Stress tests and efficiency

In [ ]:
!python scripts/evaluate.py given --dataset contrast --config configs/default.yaml
!python scripts/evaluate.py given --dataset climate  --config configs/default.yaml
!python scripts/evaluate.py given --dataset french   --config configs/default.yaml
# same French set with a multilingual verifier (zero-shot) and with the rules
!python scripts/evaluate.py given --dataset french --config configs/default.yaml --verifier-model MoritzLaurer/mDeBERTa-v3-base-mnli-xnli --tag multilingual
# (the rule-based baseline on these sets was already run on CPU: results/given_*_lexical.json)
!python scripts/evaluate.py efficiency --config configs/default.yaml
backup()

## 9b. Extension: sentences written by LLMs (LLM-AggreFact)

Real summaries and RAG answers from several LLMs, each sentence labelled supported / not supported by humans.
100 claims per sub-dataset to stay inside the Colab quota. Metric: balanced accuracy (the benchmark's own metric).
I compare my fine-tuned verifier with the zero-shot NLI checkpoint and with the rules.

**The dataset is gated.** Once: create a Hugging Face account, open https://huggingface.co/datasets/lytang/LLM-AggreFact
and accept the conditions, then create a *read* token at https://huggingface.co/settings/tokens. The next cell asks for it.

In [ ]:
from huggingface_hub import login
login()   # paste the read token; it stays in this Colab session only

In [ ]:
!python scripts/evaluate.py aggrefact --config configs/default.yaml --per-dataset 100
!python scripts/evaluate.py aggrefact --config configs/default.yaml --per-dataset 100 --verifier-model MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli --tag zeroshot
!python scripts/evaluate.py aggrefact --config configs/lexical_baseline.yaml --per-dataset 100 --tag lexical
backup()

## 10. Error export, tables, figures

In [ ]:
!python scripts/evaluate.py errors --config configs/default.yaml
!python scripts/make_tables.py
!python scripts/make_figures.py
backup()
from IPython.display import Markdown
Markdown(open("results/tables.md").read())

## 11. Save everything (results + the trained verifier)

In [ ]:
!zip -q -r results.zip results models/calibration.json models/verifier/train_log.json
!zip -q -r verifier.zip models/verifier
from google.colab import files
files.download("results.zip")
# files.download("verifier.zip")   # about 700 MB, or copy it to Google Drive instead

## 12. Try the demo from Colab

In [ ]:
# prints a public gradio.live link (stop the cell to close the demo)
!GRADIO_SHARE=1 python app.py